# 05 — Train, Compare and Tune Machine-Learning Models

This notebook performs:

- stratified 80/20 train-test split
- Case 1: baseline Bag-of-Words vs TF-IDF
- Case 2: tune the vectorizer for each model
- Case 3: tune each ML model using its best vectorizer
- save candidate tuned pipelines and the fixed test split

Models:

- Multinomial Naive Bayes
- Logistic Regression
- Linear Support Vector Machine

The optimisation metric is **macro F1**.

> **Repair from the pasted code:** the original code referenced `tfidf_case2_results` and `tfidf_grids` without defining them. This notebook includes the missing TF-IDF GridSearchCV block so the workflow is executable.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

PROJECT_ROOT = Path("..")
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "07_model_ready.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

df = df.dropna(subset=["processed_text", "sentiment"]).copy()
df = df[df["processed_text"].astype(str).str.strip().ne("")].reset_index(drop=True)

print("Rows:", len(df))
print(df["sentiment"].value_counts())
display(df.head())

## Train-test split

In [ ]:
X_text = df["processed_text"].astype(str)
y = df["sentiment"].astype(str)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training size:", len(X_train_text))
print("Testing size:", len(X_test_text))
print("\nTraining labels:")
print(y_train.value_counts())
print("\nTesting labels:")
print(y_test.value_counts())

## Metric helper

In [ ]:
def get_scores(model_name, vectorizer_name, case_name, y_true, y_pred):
    return {
        "Case": case_name,
        "Vectorizer": vectorizer_name,
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        )
    }

## Case 1 — Baseline Bag-of-Words vs TF-IDF

In [ ]:
case1_results = []

vectorizers = {
    "Bag of Words": CountVectorizer(),
    "TF-IDF": TfidfVectorizer()
}

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Support Vector Machine": LinearSVC(
        class_weight="balanced"
    )
}

for vectorizer_name, vectorizer in vectorizers.items():
    for model_name, model in models.items():

        pipe = Pipeline([
            ("vectorizer", vectorizer),
            ("model", model)
        ])

        pipe.fit(X_train_text, y_train)
        pred = pipe.predict(X_test_text)

        case1_results.append(
            get_scores(
                model_name,
                vectorizer_name,
                "Case 1 Baseline",
                y_test,
                pred
            )
        )

case1_results_df = (
    pd.DataFrame(case1_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

display(case1_results_df)

## Case 2 — Tune vectorizers only

In [ ]:
fixed_models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Support Vector Machine": LinearSVC(
        class_weight="balanced"
    )
}

bow_vectorizer_params = {
    "vectorizer__max_features": [3000, 5000, 8000],
    "vectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "vectorizer__min_df": [1, 2, 3],
    "vectorizer__max_df": [0.85, 0.90, 1.0]
}

tfidf_vectorizer_params = {
    "vectorizer__max_features": [3000, 5000, 8000],
    "vectorizer__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "vectorizer__min_df": [1, 2, 3],
    "vectorizer__max_df": [0.85, 0.90, 1.0],
    "vectorizer__sublinear_tf": [False, True]
}

### Tune Bag-of-Words

In [ ]:
bow_grids = {}
bow_case2_results = []

for model_name, model in fixed_models.items():
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer()),
        ("model", model)
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=bow_vectorizer_params,
        scoring="f1_macro",
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train_text, y_train)
    pred = grid.predict(X_test_text)

    bow_grids[model_name] = grid

    bow_case2_results.append(
        get_scores(
            model_name,
            "Bag of Words Tuned",
            "Case 2 Tune Vectorizer Only",
            y_test,
            pred
        )
    )

    print("=" * 80)
    print("Model:", model_name)
    print("Best BOW parameters:", grid.best_params_)
    print("Best CV Macro F1:", grid.best_score_)

### Tune TF-IDF

In [ ]:
tfidf_grids = {}
tfidf_case2_results = []

for model_name, model in fixed_models.items():
    pipeline = Pipeline([
        ("vectorizer", TfidfVectorizer()),
        ("model", model)
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=tfidf_vectorizer_params,
        scoring="f1_macro",
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train_text, y_train)
    pred = grid.predict(X_test_text)

    tfidf_grids[model_name] = grid

    tfidf_case2_results.append(
        get_scores(
            model_name,
            "TF-IDF Tuned",
            "Case 2 Tune Vectorizer Only",
            y_test,
            pred
        )
    )

    print("=" * 80)
    print("Model:", model_name)
    print("Best TF-IDF parameters:", grid.best_params_)
    print("Best CV Macro F1:", grid.best_score_)

### Compare Case 2 results

In [ ]:
case2_results_df = (
    pd.DataFrame(
        bow_case2_results + tfidf_case2_results
    )
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

display(case2_results_df)

## Select best vectorizer for each model

In [ ]:
best_vectorizer_setting_each_model = []

for model_name in fixed_models.keys():

    bow_row = case2_results_df[
        (case2_results_df["Model"] == model_name) &
        (case2_results_df["Vectorizer"] == "Bag of Words Tuned")
    ].iloc[0]

    tfidf_row = case2_results_df[
        (case2_results_df["Model"] == model_name) &
        (case2_results_df["Vectorizer"] == "TF-IDF Tuned")
    ].iloc[0]

    if bow_row["Macro F1"] >= tfidf_row["Macro F1"]:
        best_name = "Bag of Words Tuned"
        best_grid = bow_grids[model_name]
    else:
        best_name = "TF-IDF Tuned"
        best_grid = tfidf_grids[model_name]

    best_vectorizer_setting_each_model.append({
        "Model": model_name,
        "Best Vectorizer": best_name,
        "Best Vectorizer Parameters": best_grid.best_params_,
        "Best CV Macro F1": best_grid.best_score_
    })

best_vectorizer_setting_each_model_df = pd.DataFrame(
    best_vectorizer_setting_each_model
)

display(best_vectorizer_setting_each_model_df)

In [ ]:
def get_best_vectorizer_for_model(model_name):
    row = best_vectorizer_setting_each_model_df[
        best_vectorizer_setting_each_model_df["Model"] == model_name
    ].iloc[0]

    vectorizer_name = row["Best Vectorizer"]
    params = row["Best Vectorizer Parameters"]

    common = {
        "max_features": params["vectorizer__max_features"],
        "ngram_range": params["vectorizer__ngram_range"],
        "min_df": params["vectorizer__min_df"],
        "max_df": params["vectorizer__max_df"]
    }

    if vectorizer_name == "Bag of Words Tuned":
        vectorizer = CountVectorizer(**common)
    else:
        vectorizer = TfidfVectorizer(
            **common,
            sublinear_tf=params["vectorizer__sublinear_tf"]
        )

    return vectorizer_name, vectorizer, params

## Case 3 — Tune each ML model using its best vectorizer

### Naive Bayes

In [ ]:
nb_vectorizer_name, nb_best_vectorizer, nb_vectorizer_params = (
    get_best_vectorizer_for_model("Naive Bayes")
)

nb_pipeline = Pipeline([
    ("vectorizer", nb_best_vectorizer),
    ("model", MultinomialNB())
])

nb_model_grid = GridSearchCV(
    estimator=nb_pipeline,
    param_grid={
        "model__alpha": [0.01, 0.1, 0.5, 1.0]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

nb_model_grid.fit(X_train_text, y_train)

print("Best vectorizer:", nb_vectorizer_name)
print("Best model params:", nb_model_grid.best_params_)
print("Best CV Macro F1:", nb_model_grid.best_score_)

### Logistic Regression

In [ ]:
lr_vectorizer_name, lr_best_vectorizer, lr_vectorizer_params = (
    get_best_vectorizer_for_model("Logistic Regression")
)

lr_pipeline = Pipeline([
    ("vectorizer", lr_best_vectorizer),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

lr_model_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["liblinear", "lbfgs"]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

lr_model_grid.fit(X_train_text, y_train)

print("Best vectorizer:", lr_vectorizer_name)
print("Best model params:", lr_model_grid.best_params_)
print("Best CV Macro F1:", lr_model_grid.best_score_)

### Support Vector Machine

In [ ]:
svm_vectorizer_name, svm_best_vectorizer, svm_vectorizer_params = (
    get_best_vectorizer_for_model("Support Vector Machine")
)

svm_pipeline = Pipeline([
    ("vectorizer", svm_best_vectorizer),
    ("model", LinearSVC(class_weight="balanced"))
])

svm_model_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1, 10]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

svm_model_grid.fit(X_train_text, y_train)

print("Best vectorizer:", svm_vectorizer_name)
print("Best model params:", svm_model_grid.best_params_)
print("Best CV Macro F1:", svm_model_grid.best_score_)

## Evaluate Case 3 on the held-out test set

In [ ]:
candidate_grids = {
    "Naive Bayes Tuned": (
        nb_model_grid,
        nb_vectorizer_name,
        nb_vectorizer_params
    ),
    "Logistic Regression Tuned": (
        lr_model_grid,
        lr_vectorizer_name,
        lr_vectorizer_params
    ),
    "Support Vector Machine Tuned": (
        svm_model_grid,
        svm_vectorizer_name,
        svm_vectorizer_params
    )
}

case3_results = []

for model_name, (grid, vectorizer_name, vectorizer_params) in candidate_grids.items():
    pred = grid.predict(X_test_text)

    row = get_scores(
        model_name,
        vectorizer_name,
        "Case 3 Tune ML Model",
        y_test,
        pred
    )

    row["Best Vectorizer Parameters"] = str(vectorizer_params)
    row["Best Model Parameters"] = str(grid.best_params_)
    row["Best CV Macro F1"] = grid.best_score_

    case3_results.append(row)

case3_results_df = (
    pd.DataFrame(case3_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

display(case3_results_df)

## Save results, candidate models and exact test split

In [ ]:
all_results_df = pd.concat(
    [
        case1_results_df,
        case2_results_df,
        case3_results_df
    ],
    ignore_index=True
).sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

case1_results_df.to_csv(
    RESULTS_DIR / "case1_baseline_results.csv",
    index=False
)
case2_results_df.to_csv(
    RESULTS_DIR / "case2_vectorizer_tuning_results.csv",
    index=False
)
case3_results_df.to_csv(
    RESULTS_DIR / "case3_model_tuning_results.csv",
    index=False
)
all_results_df.to_csv(
    RESULTS_DIR / "all_model_results.csv",
    index=False
)

# Preserve the exact held-out test rows for Notebook 06.
test_split_df = pd.DataFrame({
    "processed_text": X_test_text.values,
    "sentiment": y_test.values
})
test_split_df.to_csv(
    RESULTS_DIR / "test_split.csv",
    index=False
)

joblib.dump(
    nb_model_grid.best_estimator_,
    MODELS_DIR / "naive_bayes_tuned_pipeline.pkl"
)
joblib.dump(
    lr_model_grid.best_estimator_,
    MODELS_DIR / "logistic_regression_tuned_pipeline.pkl"
)
joblib.dump(
    svm_model_grid.best_estimator_,
    MODELS_DIR / "svm_tuned_pipeline.pkl"
)

print("Saved model-training outputs.")
display(all_results_df.head(10))